In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold

import os
from dotenv import load_dotenv
# Load variables from .env file
load_dotenv()

# Get the ROOT_DIR variable
data_base_path = os.getenv("DATA_DIR")
project_root_dir = os.getenv("ROOT_DIR")



## Load the data after preprocess step from the data preparation ipynb file
The segmentation map are generated in the prerpocess step and flower and fruits number and size feature are saved in csv files.
Now using those csv files we calculate the scores for the linear regression model

In [2]:
project_root_dir

'/home2/pronoy.patra/Segmango_project/segmango_ssh'

## Prepare data where the final visit will be our ground truth
from all the visit pair data, filter out the data where the final visit will be in the output for both 2024 and 2025 dataset

In [3]:
# Load CSV
df = pd.read_csv(f"{project_root_dir}/data/mlp_all_data_with_time_weather_scale_treewise_2024.csv")

# Replace with your actual column name
column_name = 'image_name_o'

# Generate the list of 96 filenames
target_filenames = [f"08_{i:02d}_{j:02d}" for i in range(1, 13) for j in range(1, 9)]

# Filter rows where the column matches one of the target filenames
filtered_df_2024 = df[df[column_name].isin(target_filenames)].reset_index(drop=True)

# View or save
print(filtered_df_2024)
# filtered_df.to_csv("filtered_96_rows.csv", index=False)


    image_name image_name_o  n_flower   avg_flower  n_fruit    avg_fruit  \
0     01_01_01     08_01_01       263  3642.771863        7   532.857143   
1     01_01_02     08_01_02       231  4174.268398        7   695.428571   
2     01_01_03     08_01_03       112  3977.392857        6  1524.500000   
3     01_01_04     08_01_04       104  3873.884615        6   443.833333   
4     01_01_05     08_01_05        98  4093.438776        2    25.000000   
..         ...          ...       ...          ...      ...          ...   
667   07_12_04     08_12_04        22   644.181818       68  7530.058824   
668   07_12_05     08_12_05         9   512.333333       65  5120.707692   
669   07_12_06     08_12_06         4   637.250000       77  6500.103896   
670   07_12_07     08_12_07         3   256.000000       54  9155.796296   
671   07_12_08     08_12_08         4   484.000000       57  8667.385965   

     scale_sum_r  scale_max_r  scale_std_r  scale_p_10  ...       dew  \
0       1.0228

In [4]:

# Load CSV
df = pd.read_csv(f"{project_root_dir}/data/mlp_all_data_with_time_weather_scale_treewise_2025.csv")

# Replace with your actual column name
column_name = 'image_name_o'

# Generate the list of 96 filenames
target_filenames = [f"N_03_{i:02d}_{j:02d}" for i in range(1, 21) for j in range(1, 9)]

# Filter rows where the column matches one of the target filenames
filtered_df_2025 = df[df[column_name].isin(target_filenames)].reset_index(drop=True)

# View or save
print(filtered_df_2025)
# filtered_df.to_csv("filtered_96_rows.csv", index=False)


     image_name image_name_o  n_flower   avg_flower  n_fruit    avg_fruit  \
0    N_01_01_01   N_03_01_01       100  1516.470000       17   854.588235   
1    N_01_01_02   N_03_01_02       132  1361.446970       35   845.200000   
2    N_01_01_03   N_03_01_03       140  2402.928571       44  1789.863636   
3    N_01_01_04   N_03_01_04       122  1813.286885       35  1257.342857   
4    N_01_01_05   N_03_01_05       150  1511.186667       25  1338.000000   
..          ...          ...       ...          ...      ...          ...   
315  N_02_20_04   N_03_20_04         3   319.666667       48  6906.395833   
316  N_02_20_05   N_03_20_05        12  1221.583333       40  8607.600000   
317  N_02_20_06   N_03_20_06        20  1174.900000       39  6124.435897   
318  N_02_20_07   N_03_20_07         4   609.000000       35  8688.485714   
319  N_02_20_08   N_03_20_08         9  1727.555556       53  7925.339623   

     scale_sum_r  scale_max_r  scale_std_r  scale_p_10  ...       dew  \
0 

## Train and Test Split, 5 fold data preparation

In [9]:
import pandas as pd
import numpy as np

# 1. Extraction Helper: Get Tree IDs based on year-specific naming conventions
# 2024: First two chars (e.g., '04')
# 2025: Index 2 and 3 (e.g., 'N_02' -> '02')
filtered_df_2024['tree_id'] = filtered_df_2024['image_name'].str[3:5]
filtered_df_2025['tree_id'] = filtered_df_2025['image_name'].str[5:7]

# Get unique tree lists
trees_24 = filtered_df_2024['tree_id'].unique()
trees_25 = filtered_df_2025['tree_id'].unique()
print(f"Unique Trees in 2024: {trees_24}")
print(f"Unique Trees in 2025: {trees_25}")
# 2. Test Split (Randomly select 2 from 2024, 3 from 2025)
test_trees_24 = np.random.choice(trees_24, 2, replace=False)
test_trees_25 = np.random.choice(trees_25, 3, replace=False)

test_df = pd.concat([
    filtered_df_2024[filtered_df_2024['tree_id'].isin(test_trees_24)],
    filtered_df_2025[filtered_df_2025['tree_id'].isin(test_trees_25)]
])
test_df.to_csv(f"{project_root_dir}/data/train_test_splits/test_split.csv", index=False)

# 3. Prepare remaining trees for 5-Fold Cross Validation
rem_trees_24 = [t for t in trees_24 if t not in test_trees_24]
rem_trees_25 = [t for t in trees_25 if t not in test_trees_25]

# Shuffle remaining to ensure randomness across folds
np.random.shuffle(rem_trees_24)
np.random.shuffle(rem_trees_25)

print(f"--- Global Test Set ---")
print(f"2024 Test Trees: {test_trees_24}")
print(f"2025 Test Trees: {test_trees_25}\n")

# 4. 5-Fold Split (1 tree from 2024 and 2 trees from 2025 for Val per fold)
for i in range(5):
    # Slice trees for the current Validation fold
    val_24_fold = rem_trees_24[i*1 : (i+1)*1]
    val_25_fold = rem_trees_25[i*2 : (i+1)*2]
    
    # Training trees are those not in the current Validation fold
    train_24_fold = [t for t in rem_trees_24 if t not in val_24_fold]
    train_25_fold = [t for t in rem_trees_25 if t not in val_25_fold]
    
    # Create DataFrames
    val_df = pd.concat([
        filtered_df_2024[filtered_df_2024['tree_id'].isin(val_24_fold)],
        filtered_df_2025[filtered_df_2025['tree_id'].isin(val_25_fold)]
    ])
    
    train_df = pd.concat([
        filtered_df_2024[filtered_df_2024['tree_id'].isin(train_24_fold)],
        filtered_df_2025[filtered_df_2025['tree_id'].isin(train_25_fold)]
    ])
    
    # Save files
    val_df.to_csv(f"{project_root_dir}/data/train_test_splits/val_split_{i+1}.csv", index=False)
    train_df.to_csv(f"{project_root_dir}/data/train_test_splits/train_split_{i+1}.csv", index=False)
    
    print(f"--- Fold {i+1} ---")
    print(f"Val Trees 2024: {val_24_fold} | Val Trees 2025: {val_25_fold}")
    print(f"Train size: {len(train_df)} rows | Val size: {len(val_df)} rows\n")

Unique Trees in 2024: ['01' '02' '03' '04' '05' '06' '07' '08' '09' '10' '11' '12']
Unique Trees in 2025: ['01' '02' '03' '04' '05' '06' '07' '08' '09' '10' '11' '12' '13' '14'
 '15' '16' '17' '18' '19' '20']
--- Global Test Set ---
2024 Test Trees: ['11' '08']
2025 Test Trees: ['13' '06' '07']

--- Fold 1 ---
Val Trees 2024: ['05'] | Val Trees 2025: ['01', '03']
Train size: 744 rows | Val size: 88 rows

--- Fold 2 ---
Val Trees 2024: ['04'] | Val Trees 2025: ['20', '02']
Train size: 744 rows | Val size: 88 rows

--- Fold 3 ---
Val Trees 2024: ['10'] | Val Trees 2025: ['15', '17']
Train size: 744 rows | Val size: 88 rows

--- Fold 4 ---
Val Trees 2024: ['02'] | Val Trees 2025: ['16', '05']
Train size: 744 rows | Val size: 88 rows

--- Fold 5 ---
Val Trees 2024: ['09'] | Val Trees 2025: ['12', '08']
Train size: 744 rows | Val size: 88 rows



## 5-fold for LR

In [11]:
fold = 1
df_test = pd.read_csv(f'{project_root_dir}/data/train_test_splits/test_split.csv')
df_train = pd.read_csv(f'{project_root_dir}/data/train_test_splits/train_split_{fold}.csv')
df_val = pd.read_csv(f'{project_root_dir}/data/train_test_splits/val_split_{fold}.csv')
# Set random seed
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Concatenate data
data = pd.concat([df_train, df_val]).reset_index(drop=True)

features = [
    'n_flower','avg_flower','n_fruit','avg_fruit' ,
    'time',
    'temp',	'dew',	
    'precip',	
    # 'precipprob',	
    'visibility',	'solarradiation',
    'severerisk',	
    # 'preciptype',	
    'winddir',	'windgust',	'windspeed',
    'scale_sum_r_o','scale_max_r_o', 'scale_std_r_o',
    # 'scale_p_20_o'

        #   'scale_sum_r', 'scale_max_r', 'scale_std_r',
        #   'temp','dew','precip','precipprob','visibility','solarradiation','severerisk',
        #   'preciptype','winddir','windgust','windspeed'
    ]
# 'temp','dew','precip','precipprob','visibility','solarradiation','severerisk',
#             'preciptype','winddir','windgust','windspeed'
# Features and target
X = data[features].values

Y = data[['n_fruit_o']].values

X_test = df_test[features].values
Y_test = df_test[['n_fruit_o']].values

# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)
# Convert to tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
Y_tensor = torch.tensor(Y, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32)
# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=seed)

fold_results = []
fold_test_results = []
coefficients_s = [0 for i in range(len(features))]
for fold, (train_idx, val_idx) in enumerate(kf.split(X_tensor)):
    print(f"\n--- Fold {fold+1} ---")
    
    X_train, X_val = X_tensor[train_idx], X_tensor[val_idx]
    Y_train, Y_val = Y_tensor[train_idx], Y_tensor[val_idx]

    # Sklearn models work with numpy arrays
    X_train_np, Y_train_np = X_train.numpy(), Y_train.numpy().ravel()
    X_val_np, Y_val_np = X_val.numpy(), Y_val.numpy().ravel()
    X_test_np, Y_test_np = X_test_tensor.numpy(), Y_test_tensor.numpy().ravel()

    # Model
    # model = DummyRegressor(strategy='mean')  # You can replace with LinearRegression, etc.
    model = LinearRegression()
    model.fit(X_train_np, Y_train_np)
    coefficients = model.coef_
    print("Coefficients:", coefficients)
    coefficients_s = coefficients_s + coefficients
    # Predictions
    Y_val_pred = model.predict(X_val_np)

    # Evaluation
    r2 = r2_score(Y_val_np, Y_val_pred)
    mse = mean_squared_error(Y_val_np, Y_val_pred)
    mae = mean_absolute_error(Y_val_np, Y_val_pred)
    print('Validation:')
    print("R² Score:", r2)
    print("MSE:", mse)
    print("MAE:", mae)

    fold_results.append({
        'fold': fold+1,
        'r2': r2,
        'mse': mse,
        'mae': mae,
        'coef': getattr(model, 'coef_', None),
        'intercept': getattr(model, 'intercept_', None),
    })




        # Predictions Test
    Y_test_pred = model.predict(X_test_np)

    # Evaluation
    r2 = r2_score(Y_test_np, Y_test_pred)
    mse = mean_squared_error(Y_test_np, Y_test_pred)
    mae = mean_absolute_error(Y_test_np, Y_test_pred)
    print('Test:')
    print("R² Score:", r2)
    print("MSE:", mse)
    print("MAE:", mae)

    fold_test_results.append({
        'fold': fold+1,
        'r2': r2,
        'mse': mse,
        'mae': mae,
        'coef': getattr(model, 'coef_', None),
        'intercept': getattr(model, 'intercept_', None),
    })
# Summary of all folds
print("\n--- Cross-Validation Summary ---")
for result in fold_results:
    print(f"Fold {result['fold']}: R²={result['r2']:.4f}, MSE={result['mse']:.4f}, MAE={result['mae']:.4f}")
print("\n--- Cross-Test Summary ---")
for result in fold_test_results:
    print(f"Fold {result['fold']}: R²={result['r2']:.4f}, MSE={result['mse']:.4f}, MAE={result['mae']:.4f}")


# Compute mean and standard deviation
avg_r2_mean = np.mean([res['r2'] for res in fold_results])
avg_r2_std = np.std([res['r2'] for res in fold_results])

avg_mse_mean = np.mean([res['mse'] for res in fold_results])
avg_mse_std = np.std([res['mse'] for res in fold_results])

avg_mae_mean = np.mean([res['mae'] for res in fold_results])
avg_mae_std = np.std([res['mae'] for res in fold_results])

# Print in ± format with 4 decimal places
print(f"\nAverage R²:  {avg_r2_mean:.4f} ± {avg_r2_std:.4f}")
print(f"Average MSE: {avg_mse_mean:.4f} ± {avg_mse_std:.4f}")
print(f"Average MAE: {avg_mae_mean:.4f} ± {avg_mae_std:.4f}")


print('------------------------------------Test---------------------------------')
# Compute mean and standard deviation
avg_r2_mean = np.mean([res['r2'] for res in fold_test_results])
avg_r2_std = np.std([res['r2'] for res in fold_test_results])

avg_mse_mean = np.mean([res['mse'] for res in fold_test_results])
avg_mse_std = np.std([res['mse'] for res in fold_test_results])

avg_mae_mean = np.mean([res['mae'] for res in fold_test_results])
avg_mae_std = np.std([res['mae'] for res in fold_test_results])

# Print in ± format with 4 decimal places
print(f"\nAverage R²:  {avg_r2_mean:.4f} ± {avg_r2_std:.4f}")
print(f"Average MSE: {avg_mse_mean:.4f} ± {avg_mse_std:.4f}")
print(f"Average MAE: {avg_mae_mean:.4f} ± {avg_mae_std:.4f}")




--- Fold 1 ---
Coefficients: [ -2.9988077    0.34595656  15.299375     1.6699727  -18.893293
  11.837873    19.510763   -55.19592     36.902084   -40.683887
 -11.772841    -4.135072     0.24171034   0.6691127   -0.55004257
   0.6697564    0.15030897]
Validation:
R² Score: 0.411623568376943
MSE: 218.89906
MAE: 10.8551855
Test:
R² Score: 0.24434392679793715
MSE: 146.27802
MAE: 10.076826

--- Fold 2 ---
Coefficients: [-2.5209012e+00 -6.7990065e-02  1.3985200e+01  2.2745144e+00
 -5.3927940e+01 -1.2957437e+01  3.1150112e+01 -1.0953530e+02
  8.3471169e+01 -8.3769104e+01 -1.0037422e+01 -3.9001782e+00
  1.6211465e-01  4.3780267e-01 -3.6450621e-01  1.0584692e+00
  2.2066349e-01]
Validation:
R² Score: 0.518801748224132
MSE: 243.28064
MAE: 11.769976
Test:
R² Score: 0.20096982945060704
MSE: 154.67427
MAE: 10.52719

--- Fold 3 ---
Coefficients: [  -1.6087011    -0.66678834   14.229017      1.3769429   -46.699112
   -4.372387     31.14896    -101.10238      75.06514     -76.56757
  -11.644757     -

Average R²:  0.2569 ± 0.0447
Average MSE: 143.8414 ± 8.6623
Average MAE: 9.9865 ± 0.4086